# Outlier Detection Methods Demo

This notebook demonstrates various outlier detection methods available in the datascienceutils library.

## Methods Covered:
- Sigma deviation method
- IQR (Interquartile Range) method
- Z-score method
- Modified Z-score (MAD-based) method
- Percentile capping
- Outlier removal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datascienceutils as dsu

print(f"Using datascienceutils v{dsu.__version__}")

## 1. Generate Synthetic Data

Create a dataset with some outliers for demonstration.

In [ ]:
np.random.seed(42)

# Normal data
normal_data = np.random.normal(50, 10, 95)

# Add outliers
outliers = np.array([5, 10, 90, 95, 100])

# Combine
data = np.concatenate([normal_data, outliers])
np.random.shuffle(data)

print(f"Data shape: {data.shape}")
print(f"Mean: {data.mean():.2f}")
print(f"Std: {data.std():.2f}")
print(f"Min: {data.min():.2f}, Max: {data.max():.2f}")

## 2. Visualize Data Distribution

In [ ]:
plt.figure(figsize=(12, 4))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(data, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Data Distribution")
plt.grid(True, alpha=0.3)

# Box plot
plt.subplot(1, 2, 2)
plt.boxplot(data, vert=True)
plt.ylabel("Value")
plt.title("Box Plot")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Sigma Deviation Method

Detects outliers based on standard deviations from the mean.

In [ ]:
outliers_sigma, lower_sigma, upper_sigma = dsu.detect_outliers_sigma(data, n_sigma=2.0)

print(f"Sigma Method (n_sigma=2.0):")
print(f"  Lower bound: {lower_sigma:.2f}")
print(f"  Upper bound: {upper_sigma:.2f}")
print(f"  Number of outliers: {len(outliers_sigma)}")
print(f"  Outlier values: {sorted(data[outliers_sigma])[:10]}...")  # Show first 10

## 4. IQR Method

Uses the Interquartile Range to identify outliers.

In [ ]:
outliers_iqr, lower_iqr, upper_iqr = dsu.detect_outliers_iqr(data, k=1.5)

print(f"IQR Method (k=1.5):")
print(f"  Lower bound: {lower_iqr:.2f}")
print(f"  Upper bound: {upper_iqr:.2f}")
print(f"  Number of outliers: {len(outliers_iqr)}")
print(f"  Outlier values: {sorted(data[outliers_iqr])[:10]}...")  # Show first 10

## 5. Z-Score Method

Identifies outliers based on Z-scores.

In [ ]:
outliers_z = dsu.detect_outliers_zscore(data, threshold=2.0)

print(f"Z-Score Method (threshold=2.0):")
print(f"  Number of outliers: {len(outliers_z)}")
print(f"  Outlier values: {sorted(data[outliers_z])[:10]}...")  # Show first 10

## 6. Modified Z-Score Method

Uses Median Absolute Deviation (MAD) for robust outlier detection.

In [ ]:
outliers_mod_z = dsu.detect_outliers_modified_zscore(data, threshold=3.5)

print(f"Modified Z-Score Method (threshold=3.5):")
print(f"  Number of outliers: {len(outliers_mod_z)}")
print(f"  Outlier values: {sorted(data[outliers_mod_z])[:10]}...")  # Show first 10

## 7. Compare Methods

Visualize outliers detected by each method.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Outlier Detection Methods Comparison", fontsize=16)

methods = [
    ("Sigma (n=2.0)", outliers_sigma, lower_sigma, upper_sigma),
    ("IQR (k=1.5)", outliers_iqr, lower_iqr, upper_iqr),
    ("Z-Score (t=2.0)", outliers_z, None, None),
    ("Modified Z-Score (t=3.5)", outliers_mod_z, None, None)
]

for idx, (name, outlier_indices, lower, upper) in enumerate(methods):
    ax = axes[idx // 2, idx % 2]
    
    # Plot all data
    ax.scatter(range(len(data)), data, alpha=0.5, s=30, label="Normal")
    
    # Highlight outliers
    if len(outlier_indices) > 0:
        ax.scatter(outlier_indices, data[outlier_indices], 
                  color="red", s=50, label="Outliers", zorder=5)
    
    # Add bounds if available
    if lower is not None and upper is not None:
        ax.axhline(y=lower, color="green", linestyle="--", alpha=0.7, label="Bounds")
        ax.axhline(y=upper, color="green", linestyle="--", alpha=0.7)
    
    ax.set_xlabel("Index")
    ax.set_ylabel("Value")
    ax.set_title(f"{name} - {len(outlier_indices)} outliers")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Outlier Removal

Remove outliers detected by the IQR method.

In [ ]:
cleaned_data = dsu.remove_outliers(data, outliers_iqr)

print(f"Original data: {len(data)} points")
print(f"Cleaned data: {len(cleaned_data)} points")
print(f"Removed: {len(data) - len(cleaned_data)} points")
print(f"\nOriginal - Mean: {data.mean():.2f}, Std: {data.std():.2f}")
print(f"Cleaned  - Mean: {cleaned_data.mean():.2f}, Std: {cleaned_data.std():.2f}")

## 9. Percentile Capping

Cap outliers instead of removing them.

In [ ]:
capped_data = dsu.cap_outliers_percentile(data, lower_percentile=5.0, upper_percentile=95.0)

print(f"Original - Min: {data.min():.2f}, Max: {data.max():.2f}")
print(f"Capped   - Min: {capped_data.min():.2f}, Max: {capped_data.max():.2f}")

## 10. Before/After Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].hist(data, bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Original Data (n={len(data)})")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Frequency")
axes[0].grid(True, alpha=0.3)

# Cleaned
axes[1].hist(cleaned_data, bins=30, edgecolor="black", alpha=0.7, color="green")
axes[1].set_title(f"Cleaned Data (n={len(cleaned_data)})")
axes[1].set_xlabel("Value")
axes[1].grid(True, alpha=0.3)

# Capped
axes[2].hist(capped_data, bins=30, edgecolor="black", alpha=0.7, color="orange")
axes[2].set_title(f"Capped Data (n={len(capped_data)})")
axes[2].set_xlabel("Value")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. **Sigma Deviation** - Simple, assumes normal distribution
2. **IQR Method** - Robust, works well for skewed data
3. **Z-Score** - Similar to sigma, standardized
4. **Modified Z-Score** - Most robust, uses MAD
5. **Outlier Removal** - Removes detected outliers
6. **Percentile Capping** - Caps values at percentiles

**Recommendations**:
- Use **IQR** or **Modified Z-Score** for robust detection
- Use **Sigma/Z-Score** when data is normally distributed
- **Cap** instead of **remove** when you need to preserve sample size